# Coupling a symbolic half-step to a numerical diagnostic
### Porous-channel flow with `asymptotics` + SciPy

This tutorial shows how to weave the `asymptotics` library into a complete
investigation: we take an order equation the symbolic solver *cannot* finish,
solve **that** equation numerically with SciPy, and compare the resulting
two-term expansion against a fully numerical solution of the original problem.

The problem is the self-similar porous-channel flow
$$\varepsilon F''' + F F'' - (F')^2 = \lambda,\qquad F(0)=0,\ F'(1)=0,\ F(1)=1,$$
with $\varepsilon = 1/R$ the inverse cross-flow Reynolds number and $\lambda$ a
constant found as part of the solution.

In [ ]:
import numpy as np, sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from asymptotics import ODE
pi = np.pi

## 1. Symbolic half-steps

The leading-order equation $F_0 F_0'' - (F_0')^2 = 0$ is nonlinear and SymPy cannot
solve it, so we *supply* the known Taylor–Yuan profile $F_0=\sin(\pi y/2)$ with
`set_solution`. The library then forms the order-$\varepsilon$ equation with $F_0$
substituted. Crucially, `sol[1].ode` is a **live SymPy object** we can inspect and
manipulate, not just a formatted display.

In [ ]:
y = sp.Symbol('y')
eq = ODE("eps*F''' + F*F'' - F'**2", small_param="eps", independent="y",
         conditions=["F(0) = 0", "F'(1) = 0", "F(1) = 1"])
sol = eq.begin_expansion(order=1)
sol[0].set_solution(sp.sin(pi*y/2))     # F0 supplied
sol[1].ode                              # raw symbolic O(eps) equation for F1

## 2. Full nonlinear reference (SciPy)

We solve the original third-order nonlinear BVP numerically. The unknown constant
$\lambda$ is a nonlinear eigenvalue; the centreline symmetry condition $F''(0)=0$
closes the system. `solve_bvp` handles the unknown parameter directly.

In [ ]:
def full(eps):
    def rhs(t, Y, p):
        F, Fp, Fpp = Y; lam = p[0]
        return np.vstack([Fp, Fpp, (lam - F*Fpp + Fp**2)/eps])
    def bc(Ya, Yb, p):
        return np.array([Ya[0], Ya[2], Yb[1], Yb[0]-1.0])   # F(0)=0, F''(0)=0, F'(1)=0, F(1)=1
    t = np.linspace(0, 1, 201)
    Y0 = np.vstack([np.sin(pi*t/2), (pi/2)*np.cos(pi*t/2), -(pi/2)**2*np.sin(pi*t/2)])
    return solve_bvp(rhs, bc, t, Y0, p=[-pi**2/4], max_nodes=40000, tol=1e-9)

## 3. Solve `sol[1].ode` numerically

The reduced equation for $F_1$ is second order but inherits a third boundary
condition from the original problem, so it carries an undetermined **solvability
constant** $\lambda_1$ (the $O(\varepsilon)$ part of the eigenvalue). We
reintroduce it and solve the resulting singular linear BVP
$$\sin(\tfrac{\pi y}{2})F_1'' - \pi\cos(\tfrac{\pi y}{2})F_1'
  - \tfrac{\pi^2}{4}\sin(\tfrac{\pi y}{2})F_1
  = \tfrac{\pi^3}{8}\cos(\tfrac{\pi y}{2}) + \lambda_1,$$
with homogeneous conditions $F_1(0)=F_1(1)=F_1'(1)=0$.

In [ ]:
def f1_solve():
    def rhs(t, Y, p):
        F1, F1p = Y; l1 = p[0]
        s, cc = np.sin(pi*t/2), np.cos(pi*t/2)
        return np.vstack([F1p, (l1 + (pi**3/8)*cc + pi*cc*F1p + (pi**2/4)*s*F1)/s])
    a = 1e-4
    def bc(Ya, Yb, p): return np.array([Ya[0], Yb[0], Yb[1]])
    t = np.linspace(a, 1, 401)
    return solve_bvp(rhs, bc, t, np.zeros((2, t.size)), p=[0.0], max_nodes=60000, tol=1e-7)

f1 = f1_solve()
print("solvability constant lambda1 =", round(float(f1.p[0]), 4))

## 4. Compare the two-term expansion to the full solution

The composite $F_0 + \varepsilon F_1$ (with $F_1$ from the numerical half-step)
is compared to the full numerical solution across several $\varepsilon$.

In [ ]:
print(f"{'eps':>6} {'||F0-F||':>12} {'||F0+epsF1-F||':>16}")
rows=[]
for eps in [0.02, 0.05, 0.1, 0.2]:
    s = full(eps); t = np.linspace(1e-3, 1, 400); Ff = s.sol(t)[0]
    F0 = np.sin(pi*t/2); comp = F0 + eps*f1.sol(t)[0]
    e0 = np.max(np.abs(F0-Ff)); e1 = np.max(np.abs(comp-Ff)); rows.append((eps,e0,e1))
    print(f"{eps:>6} {e0:>12.3e} {e1:>16.3e}")

In [ ]:
eps = 0.1; s = full(eps)
t = np.linspace(0,1,300); tt = np.linspace(1e-4,1,300)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(t, s.sol(t)[0], 'k-', lw=2, label='Numerical reference')
ax[0].plot(t, np.sin(pi*t/2), 'C0--', lw=1.5, label='F0')
ax[0].plot(tt, np.sin(pi*tt/2)+eps*f1.sol(tt)[0], 'C3-.', lw=1.5, label='F0 + eps*F1')
ax[0].set_xlabel('y'); ax[0].set_ylabel('F'); ax[0].legend(); ax[0].set_title(f'eps={eps}')
ev=np.array([r[0] for r in rows])
ax[1].loglog(ev,[r[1] for r in rows],'C0o--',label='||F0 - F||')
ax[1].loglog(ev,[r[2] for r in rows],'C3s-.',label=r'$\|F_0+ arepsilon F_1-F\|_\infty$')
ax[1].set_xlabel('eps'); ax[1].set_ylabel('max error'); ax[1].legend()
plt.tight_layout(); plt.show()

## Takeaways

- `sol[1].ode` is a raw SymPy object: it can be fed to `lambdify`, `dsolve`, or a
  numerical solver, or manipulated term by term.
- Adding the numerically computed $F_1$ improves accuracy, and the improvement
  grows as $\varepsilon\to0$ (an $O(\varepsilon^2)$ composite error).
- At $\varepsilon=0.2$ the correction no longer helps — the diagnostic reveals the
  edge of the asymptotic regime. This is the kind of investigation the library is
  meant to support: symbolic hierarchy management from `asymptotics`, numerics from
  SciPy.